In [ ]:
import pyspark.sql.functions as F

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
# Variables
CATALOG = "use1_prod_artemis_catalog_3718194974443840"  #Change
SCHEMA = "tier1_raw"  #Change

flights_table = f"{CATALOG}.{SCHEMA}.drone_mission_table"
ortho_table = f"{CATALOG}.{SCHEMA}.drone_ortho_table"
plot_clipped_table = f"{CATALOG}.{SCHEMA}.drone_plot_clipped_table"

In [0]:
raw_flights_df = spark.table(flights_table)
raw_ortho_df = spark.table(ortho_table)
raw_plots_df = spark.table(plot_clipped_table)

master_keys = ['site', 'trial', 'season', 'flight_date']

ready_orthos_df = raw_ortho_df.filter(F.col("ortho_exists") == True).select(*master_keys)

ready_for_clipping_df = raw_flights_df.join(ready_orthos_df, on=master_keys, how='inner')

if "plots_exist" in raw_plots_df.columns:
    finish_df = raw_plots_df.filter(F.col("plots_exist") == True)
else:
    finish_df = raw_plots_df

missing_df = ready_for_clipping_df.join(
    finish_df,
    on=['flight_metadata_path'],
    how='left_anti'
)

total_ready = ready_for_clipping_df.count()
total_finish = finish_df.count()
total_missing = missing_df.count()

print("-" * 50)
print(f" PLOT CLIPPING INVENTORY REPORT:")
print(f"Flights with Ortomosaics (Ready to clip): {total_ready}")
print(f"Flights with plots already clipped: {total_finish}")
print(f"Flights pending plot clipping: {total_missing}")
print("-" * 50)

if total_missing > 0:
    display(missing_df)
else:
    print(" Everything is up to date! There are no pending flights for plot clipping.")

In [0]:
if total_missing > 0:
    # Extract the paths using list comprehension (Serverless compatible)
    path_list = [row[0] for row in missing_df.select('flight_metadata_path').collect()]

    # Set variables for the Job UI
    dbutils.jobs.taskValues.set(key="missing_clips", value=path_list)
    dbutils.jobs.taskValues.set(key="proceed", value="true")

    print(f" Green flag: {len(path_list)} pending flights sent to the Clipping Node.")

else:
    # Set empty variables if there is no work
    dbutils.jobs.taskValues.set(key="missing_clips", value=[])
    dbutils.jobs.taskValues.set(key="proceed", value="false")

    print(" Red flag: There are no pending flights. Stopping pipeline here.")